# Single solves
One control period's program on the five-cell zone solved by IPOPT: ours with the exact Hessian on the supports' structure, Sound-LSE and GMR in CasADi with L-BFGS. Updates of the iterate and the solve's seconds beside the paper's (`results/solves.csv`, `results/further_solves.csv`); the times are this machine's.

In [1]:
import os

os.environ["JAX_PLATFORMS"] = "cpu"
os.environ.update(OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")  # MUMPS and its BLAS on one thread, as the paper's solves ran
import jax
import numpy as np

jax.config.update("jax_enable_x64", True)
import time

from common import build, casadi_solve, ours_solve, paper, paper_rows, timed_period
from sparsemax_dstl.mpc import conjuncts as MC
from sparsemax_dstl.mpc import ipopt as MI
from sparsemax_dstl.mpc import program as MP
from sparsemax_dstl.sparse import patterns as SP
from sparsemax_dstl.tasks import dual_arm as D
from sparsemax_dstl.tasks import dual_arm_scenes as S
import jax.numpy as jnp

One printed line per solve: updates and seconds (with the structures built, ours), beside the paper's.

In [2]:
def show(table, label, out, **m):
    ups = paper_rows(table, quantity="updates", **m)[0]
    t = paper_rows(table, quantity="solve time", **m)[0]
    built = paper_rows(table, quantity="structures built", **m)
    extra = " structures built " + str(out["solver_objects"]) + " (paper " + built[0]["value"] + ")" if built else ""
    print(label, "|", out["status"], out["updates"], "updates,", round(out["seconds"], 2), "s" + extra, "| paper", ups["value"], "updates,",
          round(float(t["value"]), 2), "s")

From the previous control period's solution (the timed control period and the period before it), each solve twice and the second timed.

In [3]:
def previous(T):
    scene, conjs = build(S.five_cell_zone(T))
    k = timed_period(scene, conjs)
    m = dict(start="previous solution", horizon=T)
    print("T", T, "control period", k)
    show("solves", "  Sound-LSE, L-BFGS", casadi_solve(scene, conjs, k, "lse", twice=True), comparator="Sound-LSE", variant="L-BFGS", **m)
    show("solves", "  GMR, L-BFGS", casadi_solve(scene, conjs, k, "gmr", twice=True), comparator="GMR", variant="L-BFGS", **m)
    show("solves", "  ours", ours_solve(scene, conjs, k, twice=True), comparator="SparseMax (ours)", variant="supports' structure", **m)

In [4]:
previous(100)

T 100 control period 100



******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************



  Sound-LSE, L-BFGS | Solve_Succeeded 67 updates, 0.8 s | paper 67 updates, 0.79 s


  GMR, L-BFGS | Solve_Succeeded 75 updates, 0.91 s | paper 75 updates, 0.95 s


  ours | Solve_Succeeded 3 updates, 0.11 s | paper 3 updates, 0.12 s


In [5]:
previous(200)

T 200 control period 100


  Sound-LSE, L-BFGS | Solve_Succeeded 144 updates, 5.45 s | paper 144 updates, 5.53 s


  GMR, L-BFGS | Solve_Succeeded 135 updates, 5.32 s | paper 135 updates, 5.16 s


  ours | Solve_Succeeded 3 updates, 0.21 s | paper 3 updates, 0.22 s


CasADi's exact Hessian (one solve takes 14 GB at 100 time steps and 46 GB at 200) is run by `scripts/casadi_exact_hessian.py`; the paper's solves:

In [6]:
for T in (100, 200, 400):
    m = dict(start="previous solution", horizon=T, comparator="Sound-LSE", variant="CasADi's exact Hessian")
    print("T", T, "| paper", paper("solves", quantity="updates", **m), "updates,", round(paper("solves", quantity="solve time", **m), 2), "s")

T 100 | paper 3.0 updates, 1.19 s
T 200 | paper 3.0 updates, 4.13 s
T 400 | paper 3.0 updates, 11.27 s


Further solves: our solver with the structure widened by 2 and 5 samples and with the prescribed structure, from the previous solution and from the reference motion (each solve twice, the second timed, as in the paper).

In [7]:
ARMS = {"supports' structure": ("U", 0), "structure widened by 2 samples": ("D", 2), "structure widened by 5 samples": ("D", 5), "prescribed structure": ("P", 0)}


def further(T):
    scene, conjs = build(S.five_cell_zone(T))
    reference = SP.Reference(scene, conjs)
    k = timed_period(scene, conjs)
    print("T", T, "control period", k)
    for start, key in (("previous", "previous solution"), ("reference", "reference motion")):
        m = dict(scene="five-cell zone", start=key, horizon=T)
        if start == "reference":
            show("further_solves", "  " + key + ", Sound-LSE", casadi_solve(scene, conjs, k, "lse", start=start, twice=True), comparator="Sound-LSE", variant="L-BFGS", **m)
            show("further_solves", "  " + key + ", GMR", casadi_solve(scene, conjs, k, "gmr", start=start, twice=True), comparator="GMR", variant="L-BFGS", **m)
        for variant, (arm, d) in ARMS.items():
            if start == "previous" and arm == "U":
                continue
            out = ours_solve(scene, conjs, k, arm, d, start=start, reference=reference, twice=True)
            show("further_solves", "  " + key + ", ours, " + variant, out, comparator="SparseMax (ours)", variant=variant, **m)

In [8]:
further(100)

T 100 control period 100


  previous solution, ours, structure widened by 2 samples | Solve_Succeeded 3 updates, 0.12 s structures built 1 (paper 1) | paper 3 updates, 0.14 s


  previous solution, ours, structure widened by 5 samples | Solve_Succeeded 3 updates, 0.19 s structures built 1 (paper 1) | paper 3 updates, 0.21 s


  previous solution, ours, prescribed structure | Solve_Succeeded 3 updates, 0.4 s structures built 1 (paper 1) | paper 3 updates, 0.48 s


  reference motion, Sound-LSE | Solve_Succeeded 96 updates, 1.21 s | paper 96 updates, 1.18 s


  reference motion, GMR | Solve_Succeeded 126 updates, 1.62 s | paper 126 updates, 1.6 s


  reference motion, ours, supports' structure | Solve_Succeeded 14 updates, 0.67 s structures built 7 (paper 7) | paper 14 updates, 0.77 s


  reference motion, ours, structure widened by 2 samples | Solve_Succeeded 13 updates, 0.52 s structures built 3 (paper 3) | paper 13 updates, 0.59 s


  reference motion, ours, structure widened by 5 samples | Solve_Succeeded 13 updates, 0.79 s structures built 3 (paper 3) | paper 13 updates, 0.82 s


  reference motion, ours, prescribed structure | Solve_Succeeded 13 updates, 1.86 s structures built 1 (paper 1) | paper 13 updates, 1.95 s


In [9]:
further(200)

T 200 control period 100


  previous solution, ours, structure widened by 2 samples | Solve_Succeeded 3 updates, 0.36 s structures built 1 (paper 1) | paper 3 updates, 0.37 s


  previous solution, ours, structure widened by 5 samples | Solve_Succeeded 3 updates, 0.47 s structures built 1 (paper 1) | paper 3 updates, 0.48 s


  previous solution, ours, prescribed structure | Solve_Succeeded 3 updates, 1.24 s structures built 1 (paper 1) | paper 3 updates, 1.42 s


  reference motion, Sound-LSE | Solve_Succeeded 277 updates, 10.67 s | paper 277 updates, 11.36 s


  reference motion, GMR | Solve_Succeeded 228 updates, 8.6 s | paper 228 updates, 9.4 s


  reference motion, ours, supports' structure | Solve_Succeeded 12 updates, 1.21 s structures built 8 (paper 8) | paper 12 updates, 1.27 s


  reference motion, ours, structure widened by 2 samples | Solve_Succeeded 16 updates, 1.31 s structures built 4 (paper 4) | paper 16 updates, 1.42 s


  reference motion, ours, structure widened by 5 samples | Solve_Succeeded 16 updates, 2.18 s structures built 3 (paper 3) | paper 16 updates, 2.36 s


  reference motion, ours, prescribed structure | Solve_Succeeded 13 updates, 9.82 s structures built 1 (paper 1) | paper 13 updates, 10.37 s


The 64-cell circle at 200 time steps from the previous solution: ours with the prescribed structure against Sound-LSE (each solve twice, the second timed).

In [10]:
scene, conjs = build(S.circle(200))
k = timed_period(scene, conjs)
m = dict(scene="64-cell circle", start="previous solution", horizon=200)
print("control period", k)
show("further_solves", "  Sound-LSE, L-BFGS", casadi_solve(scene, conjs, k, "lse", twice=True), comparator="Sound-LSE", variant="L-BFGS", **m)
show("further_solves", "  ours, prescribed structure", ours_solve(scene, conjs, k, "P", reference=SP.Reference(scene, conjs), twice=True), comparator="SparseMax (ours)", variant="prescribed structure", **m)

control period 233


  Sound-LSE, L-BFGS | Solve_Succeeded 144 updates, 31.27 s | paper 144 updates, 31.39 s


  ours, prescribed structure | Solve_Succeeded 3 updates, 3.59 s structures built 1 (paper 1) | paper 3 updates, 3.66 s


The first solve after a hold-up of the right arm by 12, 16 or 20 samples at 200 time steps, cold from the plan that ignores the hold-up (at most 3,000 updates; each solve twice, the second timed, as in the paper). GMR's solves stop at that limit (`results/further_solves.csv`).

In [11]:
def cold(delay, k, T=200):
    scene, conjs = build(S.hold_up(T, delay))
    reference = SP.Reference(scene, conjs)
    m = dict(scene="five-cell zone", start="cold, after a hold-up of " + str(delay) + " samples", horizon=T)
    print("hold-up", delay, "samples, control period", k)
    show("further_solves", "  Sound-LSE", casadi_solve(scene, conjs, k, "lse", start="reference", twice=True, max_iter=3000), comparator="Sound-LSE", variant="L-BFGS", **m)
    for variant, (arm, d) in ARMS.items():
        out = ours_solve(scene, conjs, k, arm, d, start="reference", reference=reference, twice=True, max_iter=3000)
        show("further_solves", "  ours, " + variant, out, comparator="SparseMax (ours)", variant=variant, **m)

In [12]:
cold(12, 109)
cold(16, 100)
cold(20, 100)

hold-up 12 samples, control period 109


  Sound-LSE | Solve_Succeeded 191 updates, 6.92 s | paper 191 updates, 7.02 s


  ours, supports' structure | Solve_Succeeded 22 updates, 1.52 s structures built 9 (paper 9) | paper 22 updates, 1.61 s


  ours, structure widened by 2 samples | Solve_Succeeded 18 updates, 1.19 s structures built 4 (paper 4) | paper 18 updates, 1.24 s


  ours, structure widened by 5 samples | Solve_Succeeded 17 updates, 1.96 s structures built 3 (paper 3) | paper 17 updates, 2.01 s


  ours, prescribed structure | Solve_Succeeded 23 updates, 19.3 s structures built 1 (paper 1) | paper 23 updates, 20.95 s


... (8 outputs left out)

  ours, supports' structure | Solve_Succeeded 26 updates, 2.55 s structures built 21 (paper 21) | paper 26 updates, 2.67 s


  ours, structure widened by 2 samples | Solve_Succeeded 28 updates, 2.39 s structures built 9 (paper 9) | paper 28 updates, 2.58 s


  ours, structure widened by 5 samples | Solve_Succeeded 22 updates, 2.1 s structures built 5 (paper 5) | paper 22 updates, 2.14 s


  ours, prescribed structure | Solve_Succeeded 20 updates, 10.6 s structures built 1 (paper 1) | paper 20 updates, 11.55 s


The eleven control periods from sample 100 after a hold-up of 16 samples at 200 time steps: each period starts from the previous solution shifted by one sample, at the state the planning model reaches with the plan's first acceleration; its program is built on the record of the states so far and the plan after them. Each period is solved twice from the same start, the first solve compiling, and the second is timed and applied. The seconds are ours' IPOPT calls with the building of the solver objects and the declarations of the structures inside the solve, and Sound-LSE's IPOPT call; each period's program is built before the timed solve. The paper's row is the ten periods after the first.

In [13]:
def ten_periods(side, delay=16, k0=100, T=200, N=10):
    step = D.interval_map(D.H_STEP)  # called eagerly once per period, as the paper's loop did
    X = np.array(S.hold_up(T, delay).states)
    prev, ups, secs = None, [], []
    for j in range(N + 1):  # control periods (sequential: each starts from the previous solution)
        k = k0 + j
        if j:
            X[k] = np.asarray(step(jnp.asarray(X[k - 1]), jnp.asarray(prev[D.NX * T:D.NX * T + D.NU])))
        scene, conjs = build(S.hold_up(T, delay, states=X))
        if side == "ours":
            pb = MP.PeriodProgram(scene, conjs, k)
            s = MI.Solver(pb, max_iter=3000)
            if j == 0:
                args, kw = (pb.reference(),), {}
            else:
                w, lam_g, lam_x, carry = MI.shift_start(last, pb)
                args, kw = (w, lam_g, lam_x), {"warm": True, "carry": carry}
            s.solve(*args, **kw)  # compiles
            out = s.solve(*args, **kw)
            out["seconds"] = out["solve_wall_s"] + out["build_s"] + out["declare_s"]
            last, prev = s, s.x
        else:
            out = casadi_solve(scene, conjs, k, "lse", start="reference", twice=True, max_iter=3000) if j == 0 else casadi_after(scene, conjs, k, out)
            prev = out["x"]
        secs.append(out["seconds"])
        ups.append(out["updates"])
    return ups, secs

In [14]:
import casadi as ca
from sparsemax_dstl.mpc import baselines as B


def casadi_after(scene, conjs, k, last):
    p = B.Period(scene, conjs, k, "lse")
    x0, lam_g, lam_x = B.shift_start(last, last["period"].instances, p.instances, p.m, scene.T)
    solver = ca.nlpsol("period", "ipopt", p.nlp(), MI.ipopt_options("limited-memory", warm=True, max_iter=3000))
    lbx, ubx, lbg, ubg = p.bounds()
    solver(x0=x0, lbx=lbx, ubx=ubx, lbg=lbg, ubg=ubg, lam_g0=lam_g, lam_x0=lam_x)
    t = time.perf_counter()
    r = solver(x0=x0, lbx=lbx, ubx=ubx, lbg=lbg, ubg=ubg, lam_g0=lam_g, lam_x0=lam_x)
    seconds = time.perf_counter() - t
    st = solver.stats()
    return {"status": st["return_status"], "updates": int(st["iter_count"]), "objective": float(r["f"]), "seconds": seconds,
            "x": np.asarray(r["x"]).ravel(), "lam_g": np.asarray(r["lam_g"]).ravel(), "lam_x": np.asarray(r["lam_x"]).ravel(), "period": p}

In [15]:
m = dict(start="ten periods after a hold-up of 16 samples", horizon=200)
for side, comp, variant in (("ours", "SparseMax (ours)", "supports' structure"), ("lse", "Sound-LSE", "L-BFGS")):
    ups, secs = ten_periods(side)
    ref = paper_rows("solves", quantity="updates", comparator=comp, variant=variant, **m)[0]["values"]
    print(comp, "| updates of the ten periods", ups[1:], "| paper", [int(v) for v in ref.split(";")])
    print("   median seconds of the ten periods", round(float(np.median(secs[1:])), 3), "| paper", round(paper("solves", quantity="solve time", comparator=comp, variant=variant, **m), 3))

SparseMax (ours) | updates of the ten periods [5, 4, 4, 7, 3, 6, 3, 3, 3, 6] | paper [5, 4, 4, 7, 3, 6, 3, 3, 3, 6]
   median seconds of the ten periods 0.2 | paper 0.218


Sound-LSE | updates of the ten periods [117, 129, 117, 170, 112, 143, 115, 137, 135, 112] | paper [117, 129, 117, 170, 112, 143, 115, 137, 135, 112]
   median seconds of the ten periods 3.884 | paper 4.086
